# Transformer Sentence Embeddings — Local Training

## Before running
1. **Set `PROJECT_DIR`** in Cell 2 to the folder containing your project files (`train.py`, `data.py`, `models/`, `losses/`).
2. **Install dependencies**: Cell 1 will install them automatically.
3. **Run all cells** top-to-bottom.

> GPU is used automatically if available, otherwise falls back to CPU.

In [1]:
!pip install -q datasets scipy tqdm torch transformers

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

PyTorch : 2.9.1+cpu
CUDA    : False
Using device: cpu


In [ ]:
import os, sys, shutil

# ── Set this to the folder that contains train.py, data.py, models/, losses/ ──
PROJECT_DIR = '.'   # <-- change to your project folder path, e.g. r'C:\Users\you\my_project'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = os.path.abspath(PROJECT_DIR)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(f'PROJECT_DIR not found: {PROJECT_DIR}\nPlease set it to the folder containing train.py')

print(f'Project dir : {PROJECT_DIR}')

# Add project to Python path so imports work
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Create data folder inside project dir
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Data dir    : {DATA_DIR}')
print('Files found :', [f for f in os.listdir(PROJECT_DIR) if not f.startswith('.')][:20])

In [ ]:
# ── Download STS-B dataset ────────────────────────────────────────────────────
import pandas as pd

try:
    from datasets import load_dataset
    splits = [
        ('train',      os.path.join(DATA_DIR, 'stsb_train.csv')),
        ('validation', os.path.join(DATA_DIR, 'stsb_validation.csv')),
        ('test',       os.path.join(DATA_DIR, 'stsb_test.csv')),
    ]
    for split, out in splits:
        if os.path.exists(out):
            print(f'{out} already exists, skipping.')
            continue
        df = load_dataset('mteb/stsbenchmark-sts', split=split).to_pandas()
        if 'score' not in df.columns and 'similarity_score' in df.columns:
            df = df.rename(columns={'similarity_score': 'score'})
        df[['sentence1', 'sentence2', 'score']].to_csv(out, index=False)
        lo, hi = df['score'].min(), df['score'].max()
        print(f'{out}: {len(df):,} rows  score=[{lo:.1f}, {hi:.1f}]')
    print('Data ready.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'Manually place stsb_train/validation/test.csv in {DATA_DIR}/')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# Change working dir to project so train.py can find its relative imports
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')
!python train.py

In [ ]:
# ── Evaluate (semantic search metrics) ───────────────────────────────────────
# Requires best_model.pt to exist (produced by Cell 4)
!python evaluate_search.py

In [ ]:
# -- Inference demo -----------------------------------------------------------
import os, pickle
import torch
import torch.nn.functional as F

from data import get_pair_dataloaders
from models.model.transformer import Transformer

CONFIG = {
    'train_path': os.path.join(DATA_DIR, 'stsb_train.csv'),
    'val_path':   os.path.join(DATA_DIR, 'stsb_validation.csv'),
    'max_len':    128,
    'd_model':    256,
    'n_layers':   4,
    'n_heads':    4,
    'd_ff':       512,
    'pooling':    'mean',
    'save_path':  os.path.join(PROJECT_DIR, 'best_model.pt'),
    'vocab_path': os.path.join(PROJECT_DIR, 'vocab.pkl'),
}

# Load the exact vocab written by train.py to avoid size mismatches.
if os.path.exists(CONFIG['vocab_path']):
    with open(CONFIG['vocab_path'], 'rb') as _vf:
        vocab = pickle.load(_vf)
    print(f'Vocab loaded from vocab.pkl ({len(vocab):,} tokens)')
else:
    print('vocab.pkl not found -- rebuilding with min_freq=2 (must match train.py)')
    _, vocab = get_pair_dataloaders(
        train_path=CONFIG['train_path'],
        val_path=CONFIG['val_path'],
        min_freq=2,
    )

model = Transformer(
    vocab_size=len(vocab),
    d_model=CONFIG['d_model'],
    n_layers=CONFIG['n_layers'],
    n_heads=CONFIG['n_heads'],
    d_ff=CONFIG['d_ff'],
    max_len=CONFIG['max_len'],
    dropout=0.0,
    pooling=CONFIG['pooling'],
).to(DEVICE)

model.load_state_dict(torch.load(CONFIG['save_path'], map_location=DEVICE))
model.eval()
print('Model loaded.')

In [ ]:
@torch.no_grad()
def embed(text):
    cls_id = vocab.word2idx[vocab.CLS_TOKEN]
    sep_id = vocab.word2idx[vocab.SEP_TOKEN]
    ids = torch.tensor(
        [[cls_id] + vocab.encode(text)[:CONFIG['max_len'] - 2] + [sep_id]],
        dtype=torch.long
    ).to(DEVICE)
    return model.encode(ids, normalize=True)   # (1, d_model) unit vector


def similarity(t1, t2):
    return (embed(t1) * embed(t2)).sum().item()


test_pairs = [
    ('A dog is running in the park',   'A dog is playing outside'),
    ('The man is eating a sandwich',   'A person is having lunch'),
    ('A woman is singing a song',      'A lady is performing music'),
    ('The cat sat on the mat',         'A dog is running in the park'),
    ('A man is riding a bicycle',      'Someone is driving a car'),
    ('I love pizza',                   'The stock market crashed today'),
]

print(f'{"Sentence 1":<42} {"Sentence 2":<42} {"Sim":>5}')
print('-' * 95)
for s1, s2 in test_pairs:
    sim = similarity(s1, s2)
    bar = '#' * int((sim + 1) * 10)
    print(f'{s1:<42} {s2:<42} {sim:>5.3f}  {bar}')